[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-parallel_questions/examples/parallel_questions/parallel_questions.ipynb)

# Ask everything in one request

A briefing over a policy is a dozen narrow questions, not one essay. Asked separately, each call re-encodes the text. Asked together, the questions sit next to one copy of the text and do not attend to each other, so an extra question does not change the others.

This notebook times one request against the same questions one by one, and checks that the decisions match. `token_counts` shows the encoder work without running the model: `naive` is one sequence per option, `packed` is the packed rows, `cached` is the state once plus the question rows.

The state should stay inside the trained range. The default cut is 384 tokens; `usage.state_truncated` says when the tail was dropped.

Patterns: [Parallel questions](https://docs.typesafe.ai/cookbooks/parallel_questions) and [Speculative fan-out](https://docs.typesafe.ai/patterns/fan-out).


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
import time

QUESTIONS = {
    "kind": {"type": "choice", "instructions": "Что это за текст?",
             "criteria": {"policy": "правила возврата магазина",
                          "ad": "реклама товара",
                          "news": "новость",
                          "manual": "инструкция к прибору"}},
    "audience": {"type": "choice", "instructions": "Для кого написан текст?",
                 "criteria": {"buyer": "покупатель", "courier": "курьер",
                              "accountant": "бухгалтер", "child": "ребёнок"}},
    "contact": {"type": "choice", "instructions": "Как с магазином связаться по этому тексту?",
                "criteria": {"email": "по электронной почте", "phone": "по телефону",
                             "office": "прийти в офис", "chat": "через чат на сайте"}},
    "broken": {"type": "choice", "instructions": "Что магазин делает, если товар сломался сам в первую неделю?",
               "criteria": {"replace": "меняет или возвращает деньги",
                            "repair": "только чинит",
                            "refuse": "отказывает",
                            "credit": "даёт баллы на следующую покупку"}},
    "formality": {"type": "score", "instructions": "Насколько официален тон текста?",
                  "criteria": ["Совсем разговорный.", "Скорее разговорный.",
                               "Нейтральный.", "Скорее официальный.", "Канцелярский."]},
    "strictness": {"type": "score", "instructions": "Насколько жёсткие условия возврата?",
                   "criteria": ["Вернуть можно почти всё и почти всегда.",
                                "Условия мягкие, исключений мало.",
                                "Есть и срок, и заметные исключения.",
                                "Условия жёсткие, исключений много.",
                                "Вернуть практически нельзя."]},
    "refund": {"type": "noul", "instructions": "В тексте есть возврат денег или товара?"},
    "days14": {"type": "noul", "instructions": "Назван срок в 14 дней?"},
    "personal": {"type": "noul", "instructions": "Текст говорит, как поступят с персональными данными?"},
    "store_pays": {"type": "noul", "instructions": "Доставку при обмене брака оплачивает магазин?"},
    "sells_books": {"type": "noul", "instructions": "Магазин продаёт книги?"},
    "discount_blocks": {"type": "noul", "instructions": "Скидка отменяет право на возврат?"},
}


def decision(answer: dict):
    kind = answer["type"]
    if kind == "choice":
        return answer["choice"]
    if kind == "score":
        return round(answer["score"], 4)
    if kind == "noul":
        return round(answer["noul"], 4)
    return tuple(answer.get("ranking", ()))


def gap(left, right):
    if isinstance(left, float) and isinstance(right, float):
        return abs(left - right)
    return 0.0 if left == right else None


## Separate calls first, then the same questions together

The separate calls run first, on a cold state cache, so a packed call cannot warm the cache for them. `token_counts` is the structural comparison: `naive` re-encodes the text once per option, `packed` keeps one copy per row. Wall clock on a short text is a smaller effect than the token counts. Floats from the int8 ONNX graph can move in the second decimal between the two shapes; a choice has to match exactly.


In [4]:
def run(judge, rows: dict) -> None:
    state = rows["state"]
    full = {"state": state, "questions": QUESTIONS}
    counts = judge.token_counts(full)
    print("token_counts", counts)

    separate = {}
    started = time.perf_counter()
    for key, question in QUESTIONS.items():
        separate[key] = judge({"state": state, "questions": {key: question}})
    elapsed_separate = time.perf_counter() - started

    started = time.perf_counter()
    together = judge(full)
    elapsed_together = time.perf_counter() - started

    widest = 0.0
    mismatches = []
    for key in QUESTIONS:
        left = decision(together["answers"][key])
        right = decision(separate[key]["answers"][key])
        delta = gap(left, right)
        if delta is None:
            mismatches.append(key)
        else:
            widest = max(widest, delta)
        print(f"  {key:16} together {left!s:20} apart {right}")

    print(f"\n12 requests, cold: {elapsed_separate:.2f}s  "
          f"sum of milliseconds {sum(item['usage']['milliseconds'] for item in separate.values()):.0f}")
    print(f"one request:       {elapsed_together:.2f}s  usage {together['usage']}")
    print("state_truncated", together["usage"]["state_truncated"],
          "state_tokens", together["usage"]["state_tokens"])
    print(f"max abs gap on numeric answers {widest:.4f}")
    if mismatches:
        print("choice mismatches", mismatches)
    else:
        print(f"all {len(QUESTIONS)} decisions agree")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-parallel-questions/examples/parallel_questions/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


token_counts {'candidates': 38, 'state_tokens': 158, 'naive': 8110, 'packed': 1312, 'packed_rows': 3, 'cached': 996, 'cached_rows': 3}
  kind             together policy               apart policy
  audience         together buyer                apart buyer
  contact          together email                apart email
  broken           together replace              apart replace
  formality        together 2.8357               apart 2.8357
  strictness       together 0.0714               apart 0.0714
  refund           together 0.9817               apart 0.9817
  days14           together 0.9858               apart 0.9858
  personal         together 0.4495               apart 0.4495
  store_pays       together 0.7166               apart 0.7166
  sells_books      together 0.955                apart 0.955
  discount_blocks  together 0.4583               apart 0.4583

12 requests, cold: 11.57s  sum of milliseconds 11559
one request:       5.33s  usage {'state_tokens': 158, 'state_truncate